In [14]:
import pandas as pd 
df = pd.read_csv("../data/youtube_india_clean.csv")
print(df.shape)
print(df.columns)

(302786, 15)
Index(['video_title', 'video_category_id', 'video_duration',
       'video_definition', 'video_view_count', 'video_like_count',
       'video_comment_count', 'video_published_at', 'video_trending__date',
       'channel_title', 'channel_view_count', 'channel_subscriber_count',
       'channel_video_count', 'channel_have_hidden_subscribers', 'video_tags'],
      dtype='object')


In [15]:
#load dataset
df = pd.read_csv("../data/youtube_india_clean.csv")
#drop useless column
df = df.drop(columns=["Unnamed: 0"])
#saving the cleaned dataset againg with the same name
df.to_csv("../data/youtube_india_clean.csv",index=False)
print("Dataset Cleaned and saved")
#Confirming the columns
print(df.columns)

KeyError: "['Unnamed: 0'] not found in axis"

In [16]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302786 entries, 0 to 302785
Data columns (total 15 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   video_title                      302786 non-null  object 
 1   video_category_id                302780 non-null  object 
 2   video_duration                   302786 non-null  object 
 3   video_definition                 302786 non-null  object 
 4   video_view_count                 302767 non-null  float64
 5   video_like_count                 291925 non-null  float64
 6   video_comment_count              300979 non-null  float64
 7   video_published_at               302786 non-null  object 
 8   video_trending__date             302786 non-null  object 
 9   channel_title                    302786 non-null  object 
 10  channel_view_count               302777 non-null  float64
 11  channel_subscriber_count         302786 non-null  float64
 12  ch

For cleaning the missing values im filling the numeric values with median and dropping the useless text columns for now

In [17]:
# Make a copy so original df stays safe
df_model = df.copy()

# Fill missing numeric values using median
df_model["video_view_count"].fillna(df_model["video_view_count"].median(), inplace=True)
df_model["video_like_count"].fillna(df_model["video_like_count"].median(), inplace=True)
df_model["video_comment_count"].fillna(df_model["video_comment_count"].median(), inplace=True)
df_model["channel_view_count"].fillna(df_model["channel_view_count"].median(), inplace=True)

# Drop columns we won't use for ML right now (text columns)
df_model = df_model.drop(columns=[
    "video_title",
    "video_tags",
    "channel_title",
    "video_published_at",
    "video_trending__date"
])

# Check result
print(df_model.info())
df_model.head()
#video_category_id still has a few nulls (302780 not 302786)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302786 entries, 0 to 302785
Data columns (total 10 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   video_category_id                302780 non-null  object 
 1   video_duration                   302786 non-null  object 
 2   video_definition                 302786 non-null  object 
 3   video_view_count                 302786 non-null  float64
 4   video_like_count                 302786 non-null  float64
 5   video_comment_count              302786 non-null  float64
 6   channel_view_count               302786 non-null  float64
 7   channel_subscriber_count         302786 non-null  float64
 8   channel_video_count              302786 non-null  float64
 9   channel_have_hidden_subscribers  302786 non-null  bool   
dtypes: bool(1), float64(6), object(3)
memory usage: 21.1+ MB
None


C:\Users\yshc4\AppData\Local\Temp\ipykernel_6476\89310230.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model["video_view_count"].fillna(df_model["video_view_count"].median(), inplace=True)
C:\Users\yshc4\AppData\Local\Temp\ipykernel_6476\89310230.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values 

,video_category_id,video_duration,video_definition,video_view_count,video_like_count,video_comment_count,channel_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers
0,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False
1,Entertainment,PT4M58S,hd,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False
2,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False
3,Entertainment,PT4M58S,hd,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False
4,Music,PT3M51S,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False


Model 1 → RandomForestClassifier 

In [18]:
import pandas as pd
import re

# making a fresh copy
df_model = df.copy()

# fill missing numeric columns properly
df_model["video_view_count"] = df_model["video_view_count"].fillna(df_model["video_view_count"].median())
df_model["video_like_count"] = df_model["video_like_count"].fillna(df_model["video_like_count"].median())
df_model["video_comment_count"] = df_model["video_comment_count"].fillna(df_model["video_comment_count"].median())
df_model["channel_view_count"] = df_model["channel_view_count"].fillna(df_model["channel_view_count"].median())

# filling missing category with most frequent value
df_model["video_category_id"] = df_model["video_category_id"].fillna(df_model["video_category_id"].mode()[0])

# dropping text/date columns we are not using right now
df_model = df_model.drop(columns=[
    "video_title",
    "video_tags",
    "channel_title",
    "video_published_at",
    "video_trending__date"
])

# function to convert ISO duration like PT3M51S(duration) into total seconds
def convert_duration_to_seconds(duration):
    hours = 0
    minutes = 0
    seconds = 0

    h = re.search(r"(\d+)H", duration)
    m = re.search(r"(\d+)M", duration)
    s = re.search(r"(\d+)S", duration)

    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    if s:
        seconds = int(s.group(1))

    total_seconds = hours * 3600 + minutes * 60 + seconds
    return total_seconds

# creating a new numeric duration column
df_model["video_duration_seconds"] = df_model["video_duration"].apply(convert_duration_to_seconds)

# dropping old text duration column
df_model = df_model.drop(columns=["video_duration"])

# creating classification target
median_views = df_model["video_view_count"].median()
df_model["high_performance"] = (df_model["video_view_count"] > median_views).astype(int)
#created high_performance for classification model
# check result
print(df_model.info())
print("\nFirst 5 rows:")
display(df_model.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302786 entries, 0 to 302785
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   video_category_id                302786 non-null  object 
 1   video_definition                 302786 non-null  object 
 2   video_view_count                 302786 non-null  float64
 3   video_like_count                 302786 non-null  float64
 4   video_comment_count              302786 non-null  float64
 5   channel_view_count               302786 non-null  float64
 6   channel_subscriber_count         302786 non-null  float64
 7   channel_video_count              302786 non-null  float64
 8   channel_have_hidden_subscribers  302786 non-null  bool   
 9   video_duration_seconds           302786 non-null  int64  
 10  high_performance                 302786 non-null  int64  
dtypes: bool(1), float64(6), int64(2), object(2)
memory usage: 23.4+ M

,video_category_id,video_definition,video_view_count,video_like_count,video_comment_count,channel_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds,high_performance
0,Music,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1
1,Entertainment,hd,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False,298,1
2,Music,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1
3,Entertainment,hd,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False,298,1
4,Music,hd,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1


My model still cannot directly understand: video_category_id,video_definition. So iam encoding them.

In [19]:
#converting catagorical columns into the numeric columns using on-hot encoding
#get_dummies() changes text categories into 0 and 1 columns. in the below case true or false

df_encoded = pd.get_dummies(
    df_model,
    columns=["video_category_id", "video_definition"],
    drop_first=True

)
#checking the new shape
print("encoded dataset shape:",df_encoded.shape)
#seeeing first 5 rows
display(df_encoded.head())


encoded dataset shape: (302786, 23)


,video_view_count,video_like_count,video_comment_count,channel_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds,high_performance,video_category_id_Comedy,...,video_category_id_Gaming,video_category_id_Howto & Style,video_category_id_Music,video_category_id_News & Politics,video_category_id_People & Blogs,video_category_id_Pets & Animals,video_category_id_Science & Technology,video_category_id_Sports,video_category_id_Travel & Events,video_definition_sd
0,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1,False,...,False,False,True,False,False,False,False,False,False,False
1,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False,298,1,False,...,False,False,False,False,False,False,False,False,False,False
2,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1,False,...,False,False,True,False,False,False,False,False,False,False
3,40832089.0,951140.0,83563.0,4.907153e+08,794000.0,623.0,False,298,1,False,...,False,False,False,False,False,False,False,False,False,False
4,56032799.0,1058450.0,44767.0,2.693735e+11,276000000.0,21864.0,False,231,1,False,...,False,False,True,False,False,False,False,False,False,False


Preparing features (X) and target (y)
Target = high_performance
Did NOT include video_view_count in X, because the target was derived from it.

In [20]:
# Target column (what we want to predict)
y = df_encoded["high_performance"]

# Features (dropping the target and video_view_count to avoid data leakage)
X = df_encoded.drop(columns=["high_performance", "video_view_count"])

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (302786, 21)
y shape: (302786,)


In [21]:
from sklearn.model_selection import train_test_split

#80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Traininig set:", X_train.shape)
print("Test set:", X_test.shape)

Traininig set: (242228, 21)
Test set: (60558, 21)


In [22]:
from sklearn.ensemble import RandomForestClassifier

#creating the classification model
model_classifier = RandomForestClassifier(
    n_estimators=100,      #number of trees
    random_state=42,       #same result everytime
    n_jobs=-1              #use all cpu cores
)
#training the model
model_classifier.fit(X_train,y_train)

print("Classification model trained successfully!!!")

Classification model trained successfully!!!


In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#predicting on the test data
y_pred = model_classifier.predict(X_test)

#accuracy
accuracy = accuracy_score(y_test, y_pred)
print("accuracy:", accuracy)

#confusion matrix
print("\nConfusin matrix:")
print(confusion_matrix(y_test, y_pred))

#classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

accuracy: 0.991991148981142

Confusin matrix:
[[29973   214]
 [  271 30100]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     30187
           1       0.99      0.99      0.99     30371

    accuracy                           0.99     60558
   macro avg       0.99      0.99      0.99     60558
weighted avg       0.99      0.99      0.99     60558



Accuracy:
-My model predicts high vs low performing videos correctly about 99% of the time.
-Since we created high_performance from video_view_count, and your features include:(video_like_count,video_comment_count,channel_view_count)
-These are very strongly correlated with views, so the model finds the pattern easily. That’s why accuracy is very high.

Confusin matrix:
My model did:
True Negatives: 29,973
True Positives: 30,100
False Positives: 214
False Negatives: 271

Since the FP's and FN's are very small compared to TN's and TP.s its a very strong model.

Classification Report:
-Precision-How correct the positive predictions are
-Recall-How many real positives we found
-F1-score - balance between precision and recall
-my values are precision ≈ 0.99,recall ≈ 0.99,f1-score ≈ 0.99
-That means that the classifier is very consistent



In [24]:
import joblib

#saving the classification model

joblib.dump(model_classifier,"../models/high_performance_classifier.pkl")

print("Model saved in models folder!")

Model saved in models folder!


Testing the MODEL

In [26]:
import joblib

model = joblib.load("../models/high_performance_classifier.pkl")

sample = X_test.iloc[0:1]
actual = y_test.iloc[0]

prediction = model.predict(sample)

print("Actual:", actual)
print("Predicted:", prediction[0])

Actual: 1
Predicted: 1


Testing multiple rows instead of just one

In [27]:
# taking first 5 rows from the test set
samples = X_test.iloc[0:5]
# selecting 5 rows instead of 1 so we can see how the model performs on multiple examples

# actual values for those 5 rows
actual_values = y_test.iloc[0:5].values
# extracting the real labels for comparison
# .values gives a simple array like [1 0 1 1 0]

# predictions from the loaded model
predictions = model.predict(samples)
# asking the model to predict for all 5 rows at once

print("Actual values   :", actual_values)
# printing the true answers

print("Predicted values:", predictions)
# printing the model's predicted answers

Actual values   : [1 1 0 1 0]
Predicted values: [1 0 0 1 0]


Checking feature importance

In [28]:
import pandas as pd


importance = model.feature_importances_
# getting the importance score of each input feature from the Random Forest model
# higher value means that feature helped the model more in making predictions

feature_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importance
})
# creating a dataframe with two columns:
# feature = feature name
# importance = how useful that feature was to the model

feature_importance_df = feature_importance_df.sort_values(by="importance", ascending=False)
# sorting from highest importance to lowest importance
# this helps us see the strongest features at the top

print(feature_importance_df.head(10))
# printing the top 10 most important features

                               feature  importance
0                     video_like_count    0.395984
6               video_duration_seconds    0.151306
2                   channel_view_count    0.135054
1                  video_comment_count    0.126121
3             channel_subscriber_count    0.085360
4                  channel_video_count    0.061410
11            video_category_id_Gaming    0.023238
13             video_category_id_Music    0.006191
9      video_category_id_Entertainment    0.004834
10  video_category_id_Film & Animation    0.004184


likes + comments almost 50% of decision